# Install dependencies

In [2]:
import sys
import subprocess
import importlib.util

packages = {
    "PIL": "pillow",
    "imagehash": "ImageHash",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependency check completed.")

Dependency check completed.


# Imports and global safety settings

In [4]:
from pathlib import Path
from PIL import Image, ImageOps, UnidentifiedImageError
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import imagehash
import hashlib
import json
import os
import platform
import warnings
import traceback

Image.MAX_IMAGE_PIXELS = 300_000_000
warnings.simplefilter("error", Image.DecompressionBombWarning)

print("Imports completed.")

Imports completed.


# Configuration

In [5]:
INPUT_DIR = Path("/home/ssparzival/Imágenes/Ente Photos/Twitter/").expanduser().resolve()
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / f"01_dataset_audit_{RUN_ID}"

REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = RUN_DIR / "tables"
LOGS_DIR = RUN_DIR / "logs"

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff", ".avif"
}

MAX_WORKERS = max(1, min(16, (os.cpu_count() or 4)))
PHASH_THRESHOLD = 6

for directory in [PROJECT_DIR, RUN_DIR, REPORTS_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"INPUT_DIR does not exist: {INPUT_DIR}")

if not INPUT_DIR.is_dir():
    raise NotADirectoryError(f"INPUT_DIR is not a directory: {INPUT_DIR}")

print(f"Input directory: {INPUT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Max workers: {MAX_WORKERS}")

Input directory: /home/ssparzival/Imágenes/Ente Photos/Twitter
Run directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744
Max workers: 16


# Helper functions

In [6]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def path_to_posix(path):
    return Path(path).expanduser().resolve().as_posix()


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def imagehash_to_hex(value):
    if value is None:
        return None
    return str(value)


def hash_hex_to_int(value):
    if value is None or pd.isna(value):
        return None
    return int(str(value), 16)


def hamming_int(a, b):
    return (a ^ b).bit_count()


def safe_relative_path(path, root):
    try:
        return Path(path).resolve().relative_to(Path(root).resolve()).as_posix()
    except Exception:
        return Path(path).name


def scan_candidate_files(input_dir, valid_extensions):
    files = []
    for path in input_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in valid_extensions:
            files.append(path.resolve())
    return sorted(files, key=lambda p: p.as_posix().lower())


def inspect_image(path, input_dir):
    path = Path(path)
    base = {
        "path": path_to_posix(path),
        "relative_path": safe_relative_path(path, input_dir),
        "filename": path.name,
        "stem": path.stem,
        "extension": path.suffix.lower(),
        "parent": path.parent.as_posix(),
        "file_size_bytes": None,
        "status": "unknown",
        "error_type": None,
        "error_message": None,
        "format": None,
        "width": None,
        "height": None,
        "mode": None,
        "is_animated": None,
        "n_frames": None,
        "sha256": None,
        "phash": None,
        "dhash": None,
        "whash": None,
        "colorhash": None,
        "processed_at": now_iso(),
    }

    try:
        stat = path.stat()
        base["file_size_bytes"] = int(stat.st_size)

        if stat.st_size <= 0:
            base["status"] = "invalid"
            base["error_type"] = "EmptyFile"
            base["error_message"] = "File has zero bytes."
            return base

        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            base["format"] = img.format
            base["width"] = int(img.width)
            base["height"] = int(img.height)
            base["mode"] = img.mode
            base["is_animated"] = bool(getattr(img, "is_animated", False))
            base["n_frames"] = int(getattr(img, "n_frames", 1))

            rgb = ImageOps.exif_transpose(img.convert("RGB"))

            base["sha256"] = sha256_file(path)
            base["phash"] = imagehash_to_hex(imagehash.phash(rgb, hash_size=8))
            base["dhash"] = imagehash_to_hex(imagehash.dhash(rgb, hash_size=8))
            base["whash"] = imagehash_to_hex(imagehash.whash(rgb, hash_size=8))
            base["colorhash"] = imagehash_to_hex(imagehash.colorhash(rgb, binbits=3))

        base["status"] = "valid"
        return base

    except Exception as exc:
        base["status"] = "invalid"
        base["error_type"] = type(exc).__name__
        base["error_message"] = str(exc)
        return base


class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}
        self.rank = {item: 0 for item in items}

    def find(self, item):
        parent = self.parent[item]
        if parent != item:
            self.parent[item] = self.find(parent)
        return self.parent[item]

    def union(self, a, b):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        rank_a = self.rank[root_a]
        rank_b = self.rank[root_b]

        if rank_a < rank_b:
            self.parent[root_a] = root_b
        elif rank_a > rank_b:
            self.parent[root_b] = root_a
        else:
            self.parent[root_b] = root_a
            self.rank[root_a] += 1


class BKTree:
    def __init__(self, distance):
        self.distance = distance
        self.root = None

    def add(self, item):
        if self.root is None:
            self.root = [item, {}]
            return

        node = self.root
        while True:
            current = node[0]
            children = node[1]
            dist = self.distance(item, current)

            if dist in children:
                node = children[dist]
            else:
                children[dist] = [item, {}]
                return

    def query(self, item, threshold):
        if self.root is None:
            return []

        results = []
        nodes = [self.root]

        while nodes:
            current, children = nodes.pop()
            dist = self.distance(item, current)

            if dist <= threshold:
                results.append((current, dist))

            low = dist - threshold
            high = dist + threshold

            for child_dist, child in children.items():
                if low <= child_dist <= high:
                    nodes.append(child)

        return results


print("Helper functions loaded.")

Helper functions loaded.


# Scan candidate files

In [7]:
candidate_files = scan_candidate_files(INPUT_DIR, VALID_EXTENSIONS)

scan_report = {
    "input_dir": INPUT_DIR.as_posix(),
    "run_dir": RUN_DIR.as_posix(),
    "candidate_file_count": len(candidate_files),
    "valid_extensions": sorted(VALID_EXTENSIONS),
    "created_at": now_iso(),
}

with (REPORTS_DIR / "scan_report.json").open("w", encoding="utf-8") as f:
    json.dump(scan_report, f, ensure_ascii=False, indent=2)

print(f"Candidate image files found: {len(candidate_files)}")

if not candidate_files:
    raise RuntimeError("No candidate image files were found. Check INPUT_DIR and VALID_EXTENSIONS.")

Candidate image files found: 15744


# Validate images and compute hashes

In [8]:
records = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(inspect_image, path, INPUT_DIR): path
        for path in candidate_files
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Validating and hashing images"):
        path = futures[future]
        try:
            records.append(future.result())
        except Exception as exc:
            records.append({
                "path": path_to_posix(path),
                "relative_path": safe_relative_path(path, INPUT_DIR),
                "filename": path.name,
                "stem": path.stem,
                "extension": path.suffix.lower(),
                "parent": path.parent.as_posix(),
                "file_size_bytes": None,
                "status": "invalid",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "format": None,
                "width": None,
                "height": None,
                "mode": None,
                "is_animated": None,
                "n_frames": None,
                "sha256": None,
                "phash": None,
                "dhash": None,
                "whash": None,
                "colorhash": None,
                "processed_at": now_iso(),
            })

metadata_df = pd.DataFrame(records)

ordered_columns = [
    "path",
    "relative_path",
    "filename",
    "stem",
    "extension",
    "parent",
    "file_size_bytes",
    "status",
    "error_type",
    "error_message",
    "format",
    "width",
    "height",
    "mode",
    "is_animated",
    "n_frames",
    "sha256",
    "phash",
    "dhash",
    "whash",
    "colorhash",
    "processed_at",
]

metadata_df = metadata_df.reindex(columns=ordered_columns)

valid_df = metadata_df[metadata_df["status"].eq("valid")].copy()
invalid_df = metadata_df[metadata_df["status"].ne("valid")].copy()

print(f"Total candidates: {len(metadata_df)}")
print(f"Valid images: {len(valid_df)}")
print(f"Invalid images: {len(invalid_df)}")

Validating and hashing images: 100%|██████████| 15744/15744 [10:12<00:00, 25.69it/s]

Total candidates: 15744
Valid images: 15742
Invalid images: 2


# Exact duplicate groups by SHA-256

In [9]:
exact_duplicate_rows = []

if not valid_df.empty:
    sha_counts = valid_df["sha256"].value_counts(dropna=True)
    duplicate_sha_values = set(sha_counts[sha_counts > 1].index)

    group_id = 0
    for sha_value in sorted(duplicate_sha_values):
        group = valid_df[valid_df["sha256"].eq(sha_value)].sort_values("relative_path").copy()
        representative_path = group.iloc[0]["path"]

        for rank, (_, row) in enumerate(group.iterrows()):
            exact_duplicate_rows.append({
                "exact_group_id": f"exact_{group_id:06d}",
                "sha256": sha_value,
                "group_size": len(group),
                "is_representative": rank == 0,
                "representative_path": representative_path,
                "path": row["path"],
                "relative_path": row["relative_path"],
                "file_size_bytes": row["file_size_bytes"],
                "width": row["width"],
                "height": row["height"],
                "format": row["format"],
            })

        group_id += 1

exact_duplicates_df = pd.DataFrame(exact_duplicate_rows)

print(f"Exact duplicate groups: {exact_duplicates_df['exact_group_id'].nunique() if not exact_duplicates_df.empty else 0}")
print(f"Exact duplicate files involved: {len(exact_duplicates_df)}")

Exact duplicate groups: 19
Exact duplicate files involved: 39


# Perceptual duplicate candidates by pHash BK-tree

In [10]:
phash_df = valid_df[valid_df["phash"].notna()].copy()
phash_df["phash_int"] = phash_df["phash"].map(hash_hex_to_int)

items = list(zip(phash_df["path"].tolist(), phash_df["phash_int"].tolist()))

def bk_distance(item_a, item_b):
    return hamming_int(item_a[1], item_b[1])

tree = BKTree(bk_distance)
candidate_pairs = []
seen_pairs = set()

for item in tqdm(items, desc="Building pHash BK-tree"):
    tree.add(item)

for item in tqdm(items, desc="Querying near-duplicate pHash candidates"):
    matches = tree.query(item, PHASH_THRESHOLD)
    path_a, hash_a = item

    for matched_item, distance in matches:
        path_b, hash_b = matched_item

        if path_a == path_b:
            continue

        pair = tuple(sorted([path_a, path_b]))

        if pair in seen_pairs:
            continue

        seen_pairs.add(pair)

        candidate_pairs.append({
            "path_a": pair[0],
            "path_b": pair[1],
            "phash_distance": int(distance),
        })

perceptual_candidates_df = pd.DataFrame(candidate_pairs)

if not perceptual_candidates_df.empty:
    perceptual_candidates_df = perceptual_candidates_df.sort_values(
        ["phash_distance", "path_a", "path_b"]
    ).reset_index(drop=True)

print(f"Perceptual duplicate candidate pairs: {len(perceptual_candidates_df)}")

Querying near-duplicate pHash candidates: 100%|██████████| 15742/15742 [00:16<00:00, 977.84it/s] 


Perceptual duplicate candidate pairs: 40


# Perceptual duplicate groups by union-find

In [12]:
perceptual_group_rows = []

if not perceptual_candidates_df.empty:
    paths_in_pairs = sorted(set(perceptual_candidates_df["path_a"]).union(set(perceptual_candidates_df["path_b"])))
    uf = UnionFind(paths_in_pairs)

    for _, row in perceptual_candidates_df.iterrows():
        uf.union(row["path_a"], row["path_b"])

    groups = defaultdict(list)
    for path in paths_in_pairs:
        groups[uf.find(path)].append(path)

    path_to_meta = (
        valid_df
        .drop_duplicates(subset="path")
        .set_index("path")
        .to_dict(orient="index")
    )

    group_index = 0
    for _, group_paths in sorted(groups.items(), key=lambda item: (-len(item[1]), sorted(item[1])[0])):
        if len(group_paths) <= 1:
            continue

        group_paths = sorted(group_paths)
        representative_path = group_paths[0]

        for rank, path in enumerate(group_paths):
            meta = path_to_meta.get(path, {})
            perceptual_group_rows.append({
                "perceptual_group_id": f"perceptual_{group_index:06d}",
                "group_size": len(group_paths),
                "is_representative": rank == 0,
                "representative_path": representative_path,
                "path": path,
                "relative_path": meta.get("relative_path"),
                "file_size_bytes": meta.get("file_size_bytes"),
                "width": meta.get("width"),
                "height": meta.get("height"),
                "format": meta.get("format"),
                "phash": meta.get("phash"),
                "dhash": meta.get("dhash"),
                "whash": meta.get("whash"),
                "colorhash": meta.get("colorhash"),
            })

        group_index += 1

perceptual_groups_df = pd.DataFrame(perceptual_group_rows)

print(f"Perceptual duplicate groups: {perceptual_groups_df['perceptual_group_id'].nunique() if not perceptual_groups_df.empty else 0}")
print(f"Perceptual duplicate files involved: {len(perceptual_groups_df)}")

Perceptual duplicate groups: 40
Perceptual duplicate files involved: 80


# Save all first-step outputs

In [13]:
metadata_path = TABLES_DIR / "image_metadata.csv"
valid_path = TABLES_DIR / "valid_images.csv"
invalid_path = TABLES_DIR / "invalid_images.csv"
exact_duplicates_path = TABLES_DIR / "exact_duplicate_groups.csv"
perceptual_candidates_path = TABLES_DIR / "perceptual_duplicate_candidate_pairs.csv"
perceptual_groups_path = TABLES_DIR / "perceptual_duplicate_groups.csv"

metadata_df.to_csv(metadata_path, index=False)
valid_df.to_csv(valid_path, index=False)
invalid_df.to_csv(invalid_path, index=False)
exact_duplicates_df.to_csv(exact_duplicates_path, index=False)
perceptual_candidates_df.to_csv(perceptual_candidates_path, index=False)
perceptual_groups_df.to_csv(perceptual_groups_path, index=False)

summary = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "input_dir": INPUT_DIR.as_posix(),
    "run_dir": RUN_DIR.as_posix(),
    "total_candidate_files": int(len(metadata_df)),
    "valid_images": int(len(valid_df)),
    "invalid_images": int(len(invalid_df)),
    "exact_duplicate_groups": int(exact_duplicates_df["exact_group_id"].nunique()) if not exact_duplicates_df.empty else 0,
    "exact_duplicate_files_involved": int(len(exact_duplicates_df)),
    "perceptual_hash_threshold": int(PHASH_THRESHOLD),
    "perceptual_duplicate_candidate_pairs": int(len(perceptual_candidates_df)),
    "perceptual_duplicate_groups": int(perceptual_groups_df["perceptual_group_id"].nunique()) if not perceptual_groups_df.empty else 0,
    "perceptual_duplicate_files_involved": int(len(perceptual_groups_df)),
    "python": sys.version,
    "platform": platform.platform(),
    "max_workers": int(MAX_WORKERS),
    "outputs": {
        "metadata": metadata_path.as_posix(),
        "valid_images": valid_path.as_posix(),
        "invalid_images": invalid_path.as_posix(),
        "exact_duplicate_groups": exact_duplicates_path.as_posix(),
        "perceptual_duplicate_candidate_pairs": perceptual_candidates_path.as_posix(),
        "perceptual_duplicate_groups": perceptual_groups_path.as_posix(),
    },
}

summary_path = REPORTS_DIR / "summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved outputs:")
for key, value in summary["outputs"].items():
    print(f"{key}: {value}")

print(f"summary: {summary_path.as_posix()}")

Saved outputs:
metadata: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744/tables/image_metadata.csv
valid_images: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744/tables/valid_images.csv
invalid_images: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744/tables/invalid_images.csv
exact_duplicate_groups: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744/tables/exact_duplicate_groups.csv
perceptual_duplicate_candidate_pairs: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/01_dataset_audit_20260521_172744/tables/perceptual_duplicate_candidate_pairs

# Inspect summary

In [14]:
summary_df = pd.DataFrame([{
    "total_candidate_files": summary["total_candidate_files"],
    "valid_images": summary["valid_images"],
    "invalid_images": summary["invalid_images"],
    "exact_duplicate_groups": summary["exact_duplicate_groups"],
    "exact_duplicate_files_involved": summary["exact_duplicate_files_involved"],
    "perceptual_duplicate_candidate_pairs": summary["perceptual_duplicate_candidate_pairs"],
    "perceptual_duplicate_groups": summary["perceptual_duplicate_groups"],
    "perceptual_duplicate_files_involved": summary["perceptual_duplicate_files_involved"],
    "perceptual_hash_threshold": summary["perceptual_hash_threshold"],
}])

summary_df

,total_candidate_files,valid_images,invalid_images,exact_duplicate_groups,exact_duplicate_files_involved,perceptual_duplicate_candidate_pairs,perceptual_duplicate_groups,perceptual_duplicate_files_involved,perceptual_hash_threshold
0,15744,15742,2,19,39,40,40,80,6


# Inspect invalid files, if any

In [15]:
if invalid_df.empty:
    print("No invalid files detected.")
else:
    invalid_df[[
        "relative_path",
        "extension",
        "file_size_bytes",
        "error_type",
        "error_message",
    ]].head(30)

# Inspect exact duplicates, if any

In [16]:
if exact_duplicates_df.empty:
    print("No exact duplicates detected.")
else:
    exact_duplicates_df[[
        "exact_group_id",
        "group_size",
        "is_representative",
        "relative_path",
        "file_size_bytes",
        "width",
        "height",
        "format",
    ]].head(50)

# Inspect perceptual duplicate groups, if any

In [17]:
if perceptual_groups_df.empty:
    print("No perceptual duplicate groups detected with the current threshold.")
else:
    perceptual_groups_df[[
        "perceptual_group_id",
        "group_size",
        "is_representative",
        "relative_path",
        "file_size_bytes",
        "width",
        "height",
        "format",
        "phash",
    ]].head(80)